In [ ]:
"""
phase_decomposition.py
======================
Spectral and phase decomposition of a single GPR trace, following the
methodology of Castagna et al. (2016) "Phase decomposition", Interpretation.

Edge-effect handling
--------------------
1.  Reflect-padding: the trace is mirrored at both ends by n_pad samples
    before the CWT, then trimmed back to the original length.  This pushes
    the COI boundary outside the signal, eliminating edge noise.

2.  COI boundary overlay: dashed white lines mark the time from each edge
    at which wavelet support falls to 1/e.  Data inside the COI should be
    interpreted with caution.
"""

import numpy as np
import matplotlib.pyplot as plt
import pywt


# ---------------------------------------------------------------------------
# Core computation
# ---------------------------------------------------------------------------

def phase_decomposition(
    trace,
    dt,
    f_min_GHz: float = 0.5,
    f_max_GHz: float = 5.0,
    n_scales: int = 60,
    n_theta: int = 181,
    wavelet: str = "cmor1.5-1.0",
    reflect_pad: bool = True,
):
    """
    Spectral and phase decomposition of a 1-D GPR trace.

    Parameters
    ----------
    trace : 1-D ndarray
    dt : float
        Sampling interval in nanoseconds.
    f_min_GHz, f_max_GHz : float
        Frequency band [GHz]. Default lower limit 0.5 GHz (no physical
        energy below that for a 1.5 GHz GPR system).
    n_scales : int
        Number of CWT scales (log-spaced across the frequency band).
    n_theta : int
        Number of phase angles in the gather (default 181).
    wavelet : str
        PyWavelets complex wavelet name.
    reflect_pad : bool
        If True (default), reflect-pad the trace before CWT to suppress
        edge artefacts (cone-of-influence noise) at low frequencies.

    Returns
    -------
    A : ndarray (n_scales, n_time)
    theta_2d_deg : ndarray (n_scales, n_time)
    phase_gather : ndarray (n_theta, n_time)
    freqs_GHz : ndarray (n_scales,)  -- ascending
    time_ns : ndarray (n_time,)
    theta_deg : ndarray (n_theta,)
    """
    trace = np.asarray(trace, dtype=float)
    n_time = len(trace)
    time_ns = np.arange(n_time) * dt

    # wavelet parameters
    cwavelet = pywt.ContinuousWavelet(wavelet)
    f_c = cwavelet.center_frequency
    B   = float(wavelet.split("-")[0].replace("cmor", ""))

    # scales: ascending freqs -> descending scales
    freq_axis = np.geomspace(f_min_GHz, f_max_GHz, n_scales)
    scales = f_c / (freq_axis * dt)

    # reflect-pad to suppress COI edge artefacts
    if reflect_pad:
        n_pad = int(np.ceil(2.0 * np.sqrt(B / 2.0) * scales.max()))
        n_pad = min(n_pad, n_time)
        trace_work = np.pad(trace, n_pad, mode="reflect")
    else:
        n_pad = 0
        trace_work = trace

    coeffs, freqs_GHz = pywt.cwt(trace_work, scales, wavelet, sampling_period=dt)

    if n_pad > 0:
        coeffs = coeffs[:, n_pad : n_pad + n_time]

    A             = np.abs(coeffs)
    theta_2d_deg  = np.rad2deg(np.angle(coeffs))

    R = np.trapezoid(np.real(coeffs), freqs_GHz, axis=0)
    I = np.trapezoid(np.imag(coeffs), freqs_GHz, axis=0)

    theta_deg = np.linspace(-180.0, 180.0, n_theta)
    theta_rad = np.deg2rad(theta_deg)
    phase_gather = (
        np.outer(np.cos(theta_rad), R)
        + np.outer(np.sin(theta_rad), I)
    )

    return A, theta_2d_deg, phase_gather, freqs_GHz, time_ns, theta_deg


# ---------------------------------------------------------------------------
# COI helper
# ---------------------------------------------------------------------------

def _coi_boundary(freqs_GHz, time_ns, B, f_c=1.0):
    """COI boundary time [ns] from each edge for each frequency."""
    coi = np.sqrt(B / 2.0) * f_c / freqs_GHz
    return np.clip(coi, 0, time_ns[-1] / 2.0)


# ---------------------------------------------------------------------------
# Visualisation
# ---------------------------------------------------------------------------

def plot_decomposition(
    A,
    theta_2d_deg,
    phase_gather,
    freqs_GHz,
    time_ns,
    theta_deg,
    trace=None,
    title: str = "Spectral & Phase Decomposition",
    db_clip: float = 40.0,
    show_coi: bool = True,
    wavelet: str = "cmor1.5-1.0",
):
    """
    Three-panel figure: Amplitude Spectrum | Phase Spectrum | Phase Gather.

    Parameters
    ----------
    db_clip : float
        Dynamic range in dB for the amplitude display.
    show_coi : bool
        Overlay dashed white COI boundary lines on the spectral panels.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=True)
    fig.suptitle(title, fontsize=13, fontweight="bold")

    t0, t1 = time_ns[0], time_ns[-1]
    f_left, f_right = freqs_GHz[0], freqs_GHz[-1]   # ascending
    extent_spec = [f_left, f_right, t1, t0]

    # -- Amplitude Spectrum (dB) -------------------------------------------
    ax = axes[0]
    A_norm = A / (A.max() + 1e-30)
    A_dB = np.clip(20 * np.log10(A_norm + 1e-30), -db_clip, 0.0)
    im0 = ax.imshow(
        A_dB.T, aspect="auto", origin="upper", extent=extent_spec,
        cmap="jet", vmin=-db_clip, vmax=0.0, interpolation="bilinear",
    )
    fig.colorbar(im0, ax=ax, fraction=0.046, pad=0.04).set_label("Amplitude [dB]", fontsize=9)
    ax.set_xlabel("Frequency [GHz]", fontsize=10)
    ax.set_ylabel("TWT [ns]", fontsize=10)
    ax.set_title("Amplitude Spectrum", fontsize=11)

    # -- Phase Spectrum (degrees) ------------------------------------------
    ax = axes[1]
    im1 = ax.imshow(
        theta_2d_deg.T, aspect="auto", origin="upper", extent=extent_spec,
        cmap="RdBu", vmin=-180.0, vmax=180.0, interpolation="nearest",
    )
    fig.colorbar(im1, ax=ax, fraction=0.046, pad=0.04).set_label("Phase [deg]", fontsize=9)
    ax.set_xlabel("Frequency [GHz]", fontsize=10)
    ax.set_title("Phase Spectrum", fontsize=11)

    # -- COI overlay on amplitude and phase panels -------------------------
    if show_coi:
        B_val = float(wavelet.split("-")[0].replace("cmor", ""))
        coi_t = _coi_boundary(freqs_GHz, time_ns, B_val, f_c=1.0)
        for ax_sp in (axes[0], axes[1]):
            ax_sp.plot(freqs_GHz, coi_t,               "w--", lw=1.0, alpha=0.75, label="COI boundary")
            ax_sp.plot(freqs_GHz, time_ns[-1] - coi_t, "w--", lw=1.0, alpha=0.75)

    # -- Phase Gather ------------------------------------------------------
    ax = axes[2]
    pg_clip = np.percentile(np.abs(phase_gather), 99)
    im2 = ax.imshow(
        phase_gather.T, aspect="auto", origin="upper",
        extent=[theta_deg[0], theta_deg[-1], t1, t0],
        cmap="RdBu", vmin=-pg_clip, vmax=pg_clip, interpolation="bilinear",
    )
    fig.colorbar(im2, ax=ax, fraction=0.046, pad=0.04).set_label("Amplitude", fontsize=9)
    ax.set_xlabel("Phase [deg]", fontsize=10)
    ax.set_title("Phase Gather  S'(theta, t)", fontsize=11)
    for ph in (-90, 90):
        ax.axvline(ph, color="lime", lw=1.0, ls="--", alpha=0.8)
    ax.axvline(0, color="white", lw=0.6, ls=":", alpha=0.5)
    ax.set_xticks([-180, -90, 0, 90, 180])

    plt.tight_layout()
    return fig


# ---------------------------------------------------------------------------
# Demo / standalone entry point
# ---------------------------------------------------------------------------

def _synthetic_ricker(n, dt, f0_GHz=1.5, t_peak_ns=None):
    """Ricker wavelet with added thin-layer interference."""
    if t_peak_ns is None:
        t_peak_ns = n * dt / 3
    t = np.arange(n) * dt
    u = np.pi * f0_GHz * (t - t_peak_ns)
    ricker = (1.0 - 2.0 * u**2) * np.exp(-(u**2))
    t_thin = t_peak_ns + 0.4 / f0_GHz
    u2 = np.pi * f0_GHz * (t - t_thin)
    thin_layer = 0.4 * (1.0 - 2.0 * u2**2) * np.exp(-(u2**2))
    noise = 0 #0.05 * np.random.default_rng(42).standard_normal(n)
    return ricker + thin_layer + noise


if __name__ == "__main__":
    dt = 0.004717
    n_t = 4241
    f_c_GHz = 1.0

    trace = _synthetic_ricker(n_t, dt, f0_GHz=f_c_GHz)
    A, theta_2d_deg, phase_gather, freqs_GHz, time_ns, theta_deg = phase_decomposition(
        trace, dt, f_min_GHz=0.5, f_max_GHz=5, n_scales=60, reflect_pad=True,
    )
    fig = plot_decomposition(
        A, theta_2d_deg, phase_gather, freqs_GHz, time_ns, theta_deg,
        trace=trace,
        title=f"Phase Decomposition -- synthetic GPR trace  (f_c = {f_c_GHz} GHz)",
        show_coi=True,
    )
    plt.show()


In [ ]:
# =============================================================================
# Synthetic GPR B-scan — linearly thinning fracture (constant impedance)
#
# Goal: isolate the effect of aperture on the GPR response.
#   x = 0      aperture = 2*lambda  (two full wavelengths, well-separated events)
#   x -> xmax  aperture decreases linearly to sub-Rayleigh (< lambda/4)
#
# Impedance contrast is CONSTANT along the profile so any amplitude/phase
# variation is caused purely by thin-bed interference, not by impedance change.
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Acquisition ────────────────────────────────────────────────────────────────
dt  = 0.004717      # ns
f_c = 1.5           # GHz — GPR centre frequency
n_t = 2000          # samples (~7.1 ns record)
n_x = 80            # lateral traces
dx  = 0.05          # m per trace

t = np.arange(n_t) * dt
x = np.arange(n_x) * dx

# ── Key time constants ─────────────────────────────────────────────────────────
T_lam  = 1.0 / f_c          # dominant period  = 0.667 ns  (= lambda in TWT)
T_tune = T_lam / 4.0        # Rayleigh tuning criterion = 0.167 ns

# ── Ricker wavelet ─────────────────────────────────────────────────────────────
n_wav = 251
t_wav = (np.arange(n_wav) - n_wav // 2) * dt
u_wav = np.pi * f_c * t_wav
ricker = (1.0 - 2.0 * u_wav**2) * np.exp(-u_wav**2)

# ── Fracture: constant impedance, linearly decreasing aperture ─────────────────
Z_rock  = 1.0
Z_fill  = 0.50                                     # constant fill impedance
R_top   = (Z_fill - Z_rock) / (Z_fill + Z_rock)   # = -1/3  (constant)

ap_max  = 2.0 * T_lam    # 1.333 ns  — two full wavelengths at x = 0
ap_min  = 0.015          # ~0.09*T_tune — far below Rayleigh at x[-1]
aperture = np.linspace(ap_max, ap_min, n_x)

T0 = np.full(n_x, 4.5)    # flat fracture at 4.5 ns

# Position where aperture crosses the Rayleigh criterion
i_rayleigh = int(np.argmin(np.abs(aperture - T_tune)))
x_rayleigh = x[i_rayleigh]

# ── Selected traces at physically meaningful apertures ─────────────────────────
target_ap  = [2.0*T_lam, T_lam, T_lam/2, T_tune, T_lam/8]
target_lbl = [
    f"2λ  (thick bed)",
    f"λ  (1 period)",
    f"λ/2  (peak amplitude)",
    f"λ/4  (Rayleigh limit)",
    f"λ/8  (sub-Rayleigh)",
]
pd_idx    = [int(np.argmin(np.abs(aperture - a))) for a in target_ap]
pd_labels = target_lbl
pd_colors = ["royalblue", "forestgreen", "darkorange", "crimson", "purple"]

# ── Convolutional synthesis ────────────────────────────────────────────────────
rng   = np.random.default_rng(7)
bscan = np.zeros((n_x, n_t))
for i in range(n_x):
    ref        = np.zeros(n_t)
    i_top      = min(int(round(T0[i] / dt)), n_t - 1)
    i_bot      = min(i_top + max(1, int(round(aperture[i] / dt))), n_t - 1)
    ref[i_top] =  R_top
    ref[i_bot] = -R_top
    bscan[i]   = np.convolve(ref, ricker, mode="same")

noise_level = 0.02 * np.abs(bscan).max()
bscan += noise_level * rng.standard_normal(bscan.shape)

# ── Figure: aperture profile + B-scan ─────────────────────────────────────────
fig = plt.figure(figsize=(13, 8))
gs  = gridspec.GridSpec(2, 1, height_ratios=[0.25, 1.0], hspace=0.35)

# Panel 1: aperture profile
ax0 = fig.add_subplot(gs[0])
ax0.plot(x, aperture / T_tune, "k-", lw=1.8)
ax0.axhline(1.0, color="crimson", ls="--", lw=1.2, label="Rayleigh criterion (T_tune = λ/4)")
ax0.axhline(4.0, color="royalblue", ls=":",  lw=0.9, label="2λ")
ax0.fill_between(x, aperture / T_tune, 1.0,
                 where=(aperture < T_tune), alpha=0.15, color="crimson",
                 label="sub-Rayleigh zone")
ax0.set_ylabel("Aperture / T_tune", fontsize=9)
ax0.set_title("Fracture aperture profile  (R_top = {:.3f} everywhere)".format(R_top), fontsize=10)
ax0.legend(fontsize=8, loc="upper right")
ax0.set_xlim(x[0], x[-1])
ax0.set_ylim(0, None)
for ix, col, lbl in zip(pd_idx, pd_colors, pd_labels):
    ax0.axvline(x[ix], color=col, lw=1.1, ls="--", alpha=0.8)

# Panel 2: B-scan
ax1 = fig.add_subplot(gs[1])
clip = np.percentile(np.abs(bscan), 98)
ax1.imshow(
    bscan.T, aspect="auto", origin="upper",
    extent=[x[0], x[-1], t[-1], t[0]],
    cmap="RdBu", vmin=-clip, vmax=clip, interpolation="bilinear",
)
ax1.axvline(x_rayleigh, color="crimson", lw=1.5, ls="--", alpha=0.9,
            label=f"Rayleigh limit  x = {x_rayleigh:.2f} m")
ax1.set_xlabel("Position [m]", fontsize=10)
ax1.set_ylabel("TWT [ns]", fontsize=10)
ax1.set_title("Synthetic GPR B-scan — thinning fracture (constant impedance)", fontsize=11)
for ix, col, lbl in zip(pd_idx, pd_colors, pd_labels):
    ax1.axvline(x[ix], color=col, lw=1.2, ls="--", alpha=0.9, label=lbl)
ax1.legend(fontsize=8, loc="upper right")
ax1.set_ylim(T0[0] + 1.5, T0[0] - 0.8)   # zoom to fracture zone

plt.suptitle("Thinning Fracture B-scan  —  Convolutional Model", fontsize=12,
             fontweight="bold")
plt.tight_layout()
plt.show()

# Print aperture at each selected trace
print(f"  {'Trace':<28} {'x[m]':>6} {'ap[ps]':>8} {'ap/T_tune':>10}")
print("  " + "-"*56)
for lbl, ix in zip(pd_labels, pd_idx):
    print(f"  {lbl:<28} {x[ix]:>6.2f} {aperture[ix]*1e3:>8.1f} {aperture[ix]/T_tune:>10.2f}x")


In [ ]:
# =============================================================================
# Phase decomposition of the five selected traces
# =============================================================================

for lbl, ix, col in zip(pd_labels, pd_idx, pd_colors):
    trace = bscan[ix]
    ap_ratio = aperture[ix] / T_tune

    A, theta_2d_deg, pg, freqs_GHz, time_ns, theta_deg = phase_decomposition(
        trace, dt,
        f_min_GHz=0.5, f_max_GHz=4.5, n_scales=60,
        reflect_pad=True,
    )
    fig = plot_decomposition(
        A, theta_2d_deg, pg, freqs_GHz, time_ns, theta_deg,
        title=(
            f"Phase Decomp — {lbl}  "
            f"(aperture = {aperture[ix]*1e3:.0f} ps = {ap_ratio:.2f}*T_tune)"
        ),
        show_coi=True,
    )
    plt.show()


In [ ]:
# =============================================================================
# Phase-selective B-scan reconstruction  —  ±90° (thin-bed) component
# =============================================================================

import pywt
import numpy as np
import matplotlib.pyplot as plt

# ── CWT: build R(t) and I(t) for every trace ──────────────────────────────────
wav   = "cmor1.5-1.0"
cwav  = pywt.ContinuousWavelet(wav)
f_c_w = cwav.center_frequency
B_w   = float(wav.split("-")[0].replace("cmor", ""))

scales_pd = f_c_w / (np.geomspace(0.5, 4.5, 60) * dt)
n_pad_pd  = min(int(np.ceil(2.0 * np.sqrt(B_w / 2.0) * scales_pd.max())), n_t)

bscan_R = np.zeros((n_x, n_t))
bscan_I = np.zeros((n_x, n_t))

for i in range(n_x):
    tr_pad         = np.pad(bscan[i], n_pad_pd, mode="reflect")
    coeffs, freqs_ = pywt.cwt(tr_pad, scales_pd, wav, sampling_period=dt)
    coeffs         = coeffs[:, n_pad_pd : n_pad_pd + n_t]
    bscan_R[i]     = np.trapezoid(np.real(coeffs), freqs_, axis=0)
    bscan_I[i]     = np.trapezoid(np.imag(coeffs), freqs_, axis=0)

envelope  = np.sqrt(bscan_R**2 + bscan_I**2)
bscan_sin = bscan_I / (envelope + 1e-30)

amp_threshold    = 0.01
mask             = envelope > amp_threshold * envelope.max()
bscan_sin_masked = np.where(mask, bscan_sin, np.nan)

# ── Figure 1: 5-panel B-scan comparison ───────────────────────────────────────
t_lo, t_hi = T0[0] - 0.8, T0[0] + 1.6
i_lo, i_hi = int(t_lo / dt), int(t_hi / dt)
ext         = [x[0], x[-1], t[i_hi - 1], t[i_lo]]

fig, axes = plt.subplots(1, 5, figsize=(24, 5), sharey=True, sharex=True)
fig.suptitle("Phase-selective B-scan reconstruction  (fracture time window)",
             fontsize=12, fontweight="bold")

panels = [
    (bscan,            "RdBu",  "Original B-scan"),
    (bscan_R,          "RdBu",  "theta = 0 deg   in-phase  R(t)"),
    (bscan_I,          "RdBu",  "theta = +90 deg   I(t)"),
    (-bscan_I,         "RdBu",  "theta = -90 deg   -I(t)"),
    (bscan_sin_masked, "RdBu",  f"sin(theta_dom) masked (envelope > {amp_threshold:.0%})"),
]

for ax, (data, cmap, title) in zip(axes, panels):
    vabs = 0.95 if data is bscan_sin_masked else np.percentile(np.abs(data), 98)
    im = ax.imshow(data[:, i_lo:i_hi].T, aspect="auto", origin="upper",
                   extent=ext, cmap=cmap, vmin=-vabs, vmax=vabs,
                   interpolation="bilinear")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.axvline(x_rayleigh, color="crimson", lw=1.5, ls="--", alpha=0.9)
    ax.set_xlabel("Position [m]", fontsize=9)
    ax.set_title(title, fontsize=9)
    for ix, col in zip(pd_idx, pd_colors):
        ax.axvline(x[ix], color=col, lw=0.9, ls=":", alpha=0.8)

axes[0].set_ylabel("TWT [ns]", fontsize=10)
axes[0].text(x_rayleigh + 0.05, t_lo + 0.15, "lambda/4", color="crimson", fontsize=9)
plt.tight_layout()
plt.show()

# ── Compute the TRUE tuning thickness from the noise-free Ricker pair ──────────
# Sweep aperture finely, convolve noise-free pair, record peak amplitude.
# The tuning peak is where the amplitude is maximum: this is the actual Rayleigh
# criterion for this wavelet (not necessarily exactly lambda/4).
n_wav_tc = 251
t_wav_tc  = (np.arange(n_wav_tc) - n_wav_tc // 2) * dt
ricker_tc = (1.0 - 2.0*(np.pi*f_c*t_wav_tc)**2) * np.exp(-(np.pi*f_c*t_wav_tc)**2)

deltas    = np.linspace(0, ap_max, 300)
amp_tune  = np.zeros(len(deltas))
ref_tc    = np.zeros(n_t)
i_top_tc  = int(round(T0[0] / dt))

for k, delta in enumerate(deltas):
    ref_tc[:] = 0.0
    ref_tc[i_top_tc] = R_top
    i_bot_tc = min(i_top_tc + max(1, int(round(delta / dt))), n_t - 1)
    ref_tc[i_bot_tc] = -R_top
    tr_tc = np.convolve(ref_tc, ricker_tc, mode="same")
    win_l = max(0,     i_top_tc - int(0.3 / dt))
    win_r = min(n_t-1, i_top_tc + int(1.5 * T_lam / dt))
    amp_tune[k] = np.abs(tr_tc[win_l:win_r]).max()

i_tune_peak   = np.argmax(amp_tune)
ap_tune_peak  = deltas[i_tune_peak]          # actual tuning thickness [ns]
ap_tune_ratio = ap_tune_peak / T_tune        # in units of lambda/4

# ── Figure 2: THE TUNING CURVE ─────────────────────────────────────────────────
# Use raw peak amplitude from bscan (not CWT envelope) for apples-to-apples
# comparison with the noise-free reference curve above.
t0_idx  = int(round(T0[0] / dt))
win_lo  = max(0,     t0_idx - int(0.3  / dt))
win_hi  = min(n_t-1, t0_idx + int(1.5 * T_lam / dt))

amp_raw_at_frac = np.array([np.abs(bscan[i, win_lo:win_hi]).max() for i in range(n_x)])
sin_at_frac     = np.array([
    bscan_sin_masked[i, win_lo + np.argmax(np.abs(bscan[i, win_lo:win_hi]))]
    for i in range(n_x)
])

amp_raw_norm  = amp_raw_at_frac / amp_raw_at_frac.max()
amp_tune_norm = amp_tune / amp_tune.max()
ap_ratio      = aperture / T_tune
delta_ratio   = deltas   / T_tune

fig2, ax2 = plt.subplots(figsize=(9, 5))
fig2.suptitle("The tuning curve  —  raw amplitude vs phase indicator", fontsize=12,
              fontweight="bold")

ax2.plot(delta_ratio, amp_tune_norm,       "k--", lw=1.5, alpha=0.7,
         label="Noise-free tuning curve (Ricker pair)")
ax2.plot(ap_ratio,    amp_raw_norm,        "k-",  lw=2.0,
         label="Peak amplitude at fracture (bscan)")
ax2.plot(ap_ratio,    np.abs(sin_at_frac), "m-",  lw=2.0,
         label="|sin(theta_dom)|  at fracture")

# Mark lambda/4 (theoretical) and actual tuning peak
ax2.axvline(1.0,           color="steelblue", ls="--", lw=1.5,
            label=f"lambda/4  (T_tune = {T_tune*1e3:.0f} ps, theoretical)")
ax2.axvline(ap_tune_ratio, color="crimson",   ls="-",  lw=1.5,
            label=f"Actual tuning peak = {ap_tune_ratio:.2f} * T_tune"
                  f"  ({ap_tune_peak*1e3:.0f} ps)")

ax2.fill_betweenx([0, 1.05], 0, ap_tune_ratio,
                  alpha=0.07, color="crimson", label="Sub-tuning zone")

for lbl, ix, col in zip(pd_labels, pd_idx, pd_colors):
    ax2.axvline(ap_ratio[ix], color=col, lw=1.0, ls=":", alpha=0.9)
    ax2.text(ap_ratio[ix] + 0.05, 0.88, lbl.split("(")[0].strip(),
             color=col, fontsize=8, rotation=90, va="top")

ax2.set_xlabel("Aperture / T_tune  (= aperture / [lambda/4])", fontsize=10)
ax2.set_ylabel("Normalised value", fontsize=10)
ax2.set_xlim(0, ap_ratio.max())
ax2.set_ylim(0, 1.08)
ax2.legend(fontsize=8, loc="upper right")
ax2.set_title(
    "Amplitude peaks at the actual tuning thickness (red line), "
    "not necessarily at lambda/4 (blue dashed)\n"
    "Left of red: amplitude drops, |sin(theta_dom)| stays near 1  "
    "-> phase detects beds amplitude cannot resolve",
    fontsize=9,
)
plt.tight_layout()
plt.show()

print(f"Theoretical lambda/4      : T_tune = {T_tune*1e3:.1f} ps = T_lam / 4")
print(f"Actual tuning peak (Ricker): {ap_tune_peak*1e3:.1f} ps = {ap_tune_ratio:.2f} * T_tune "
      f"= {ap_tune_peak/T_lam:.3f} * T_lam")

# ── Figure 3: Trace overlays at the five key positions ────────────────────────
fig3, axs3 = plt.subplots(1, 5, figsize=(18, 6), sharey=True)
fig3.suptitle("Composite waveform at key apertures  (normalised per trace)", fontsize=11,
              fontweight="bold")

for ax, lbl, ix, col in zip(axs3, pd_labels, pd_idx, pd_colors):
    tr   = bscan[ix]
    norm = max(np.abs(tr).max(), 1e-9)
    ax.plot(tr / norm, t, color=col, lw=1.4)
    ax.axhline(T0[ix],               color="gray", lw=0.7, ls="--")
    ax.axhline(T0[ix] + aperture[ix], color="gray", lw=0.7, ls="--")
    ax.axvline(0, color="black", lw=0.4)
    ax.invert_yaxis()
    ax.set_ylim(T0[0] + 1.6, T0[0] - 0.6)
    ax.set_xlim(-1.3, 1.3)
    ax.set_title(lbl, fontsize=9, color=col)
    ax.set_xlabel("Norm. amp.", fontsize=8)
    if ax is axs3[0]:
        ax.set_ylabel("TWT [ns]", fontsize=9)
    ax.text(0.05, 0.02, f"{aperture[ix]*1e3:.0f} ps\n= {aperture[ix]/T_tune:.2f}*lambda/4",
            transform=ax.transAxes, fontsize=8, va="bottom")

plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# Phase decomposition of the five selected traces
# =============================================================================

for lbl, ix, col in zip(pd_labels, pd_idx, pd_colors):
    trace = bscan[ix]
    ap_ratio = aperture[ix] / T_tune

    A, theta_2d_deg, pg, freqs_GHz, time_ns, theta_deg = phase_decomposition(
        trace, dt,
        f_min_GHz=0.5, f_max_GHz=4.5, n_scales=60,
        reflect_pad=True,
    )
    fig = plot_decomposition(
        A, theta_2d_deg, pg, freqs_GHz, time_ns, theta_deg,
        title=(
            f"Phase Decomp — {lbl}  "
            f"(aperture = {aperture[ix]*1e3:.0f} ps = {ap_ratio:.2f}*T_tune)"
        ),
        show_coi=True,
    )
    plt.show()


In [ ]:
# =============================================================================
# Phase-selective B-scan reconstruction  —  ±90° (thin-bed) component
# =============================================================================

import pywt
import numpy as np
import matplotlib.pyplot as plt

# ── CWT: build R(t) and I(t) for every trace ──────────────────────────────────
wav   = "cmor1.5-1.0"
cwav  = pywt.ContinuousWavelet(wav)
f_c_w = cwav.center_frequency
B_w   = float(wav.split("-")[0].replace("cmor", ""))

scales_pd = f_c_w / (np.geomspace(0.5, 4.5, 60) * dt)
n_pad_pd  = min(int(np.ceil(2.0 * np.sqrt(B_w / 2.0) * scales_pd.max())), n_t)

bscan_R = np.zeros((n_x, n_t))
bscan_I = np.zeros((n_x, n_t))

for i in range(n_x):
    tr_pad         = np.pad(bscan[i], n_pad_pd, mode="reflect")
    coeffs, freqs_ = pywt.cwt(tr_pad, scales_pd, wav, sampling_period=dt)
    coeffs         = coeffs[:, n_pad_pd : n_pad_pd + n_t]
    bscan_R[i]     = np.trapezoid(np.real(coeffs), freqs_, axis=0)
    bscan_I[i]     = np.trapezoid(np.imag(coeffs), freqs_, axis=0)

envelope  = np.sqrt(bscan_R**2 + bscan_I**2)
bscan_sin = bscan_I / (envelope + 1e-30)

amp_threshold    = 0.01
mask             = envelope > amp_threshold * envelope.max()
bscan_sin_masked = np.where(mask, bscan_sin, np.nan)

# ── Figure 1: 5-panel B-scan comparison ───────────────────────────────────────
t_lo, t_hi = T0[0] - 0.8, T0[0] + 1.6
i_lo, i_hi = int(t_lo / dt), int(t_hi / dt)
ext         = [x[0], x[-1], t[i_hi - 1], t[i_lo]]

fig, axes = plt.subplots(1, 5, figsize=(24, 5), sharey=True, sharex=True)
fig.suptitle("Phase-selective B-scan reconstruction  (fracture time window)",
             fontsize=12, fontweight="bold")

panels = [
    (bscan,            "RdBu",  "Original B-scan"),
    (bscan_R,          "RdBu",  "theta = 0 deg   in-phase  R(t)"),
    (bscan_I,          "RdBu",  "theta = +90 deg   I(t)"),
    (-bscan_I,         "RdBu",  "theta = -90 deg   -I(t)"),
    (bscan_sin_masked, "RdBu",  f"sin(theta_dom) masked (envelope > {amp_threshold:.0%})"),
]

for ax, (data, cmap, title) in zip(axes, panels):
    vabs = 0.95 if data is bscan_sin_masked else np.percentile(np.abs(data), 98)
    im = ax.imshow(data[:, i_lo:i_hi].T, aspect="auto", origin="upper",
                   extent=ext, cmap=cmap, vmin=-vabs, vmax=vabs,
                   interpolation="bilinear")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.axvline(x_rayleigh, color="crimson", lw=1.5, ls="--", alpha=0.9)
    ax.set_xlabel("Position [m]", fontsize=9)
    ax.set_title(title, fontsize=9)
    for ix, col in zip(pd_idx, pd_colors):
        ax.axvline(x[ix], color=col, lw=0.9, ls=":", alpha=0.8)

axes[0].set_ylabel("TWT [ns]", fontsize=10)
axes[0].text(x_rayleigh + 0.05, t_lo + 0.15, "lambda/4", color="crimson", fontsize=9)
plt.tight_layout()
plt.show()

# ── Compute the TRUE tuning thickness from the noise-free Ricker pair ──────────
# Sweep aperture finely, convolve noise-free pair, record peak amplitude.
# The tuning peak is where the amplitude is maximum: this is the actual Rayleigh
# criterion for this wavelet (not necessarily exactly lambda/4).
n_wav_tc = 251
t_wav_tc  = (np.arange(n_wav_tc) - n_wav_tc // 2) * dt
ricker_tc = (1.0 - 2.0*(np.pi*f_c*t_wav_tc)**2) * np.exp(-(np.pi*f_c*t_wav_tc)**2)

deltas    = np.linspace(0, ap_max, 300)
amp_tune  = np.zeros(len(deltas))
ref_tc    = np.zeros(n_t)
i_top_tc  = int(round(T0[0] / dt))

for k, delta in enumerate(deltas):
    ref_tc[:] = 0.0
    ref_tc[i_top_tc] = R_top
    i_bot_tc = min(i_top_tc + max(1, int(round(delta / dt))), n_t - 1)
    ref_tc[i_bot_tc] = -R_top
    tr_tc = np.convolve(ref_tc, ricker_tc, mode="same")
    win_l = max(0,     i_top_tc - int(0.3 / dt))
    win_r = min(n_t-1, i_top_tc + int(1.5 * T_lam / dt))
    amp_tune[k] = np.abs(tr_tc[win_l:win_r]).max()

i_tune_peak   = np.argmax(amp_tune)
ap_tune_peak  = deltas[i_tune_peak]          # actual tuning thickness [ns]
ap_tune_ratio = ap_tune_peak / T_tune        # in units of lambda/4

# ── Figure 2: THE TUNING CURVE ─────────────────────────────────────────────────
# Use raw peak amplitude from bscan (not CWT envelope) for apples-to-apples
# comparison with the noise-free reference curve above.
t0_idx  = int(round(T0[0] / dt))
win_lo  = max(0,     t0_idx - int(0.3  / dt))
win_hi  = min(n_t-1, t0_idx + int(1.5 * T_lam / dt))

amp_raw_at_frac = np.array([np.abs(bscan[i, win_lo:win_hi]).max() for i in range(n_x)])
sin_at_frac     = np.array([
    bscan_sin_masked[i, win_lo + np.argmax(np.abs(bscan[i, win_lo:win_hi]))]
    for i in range(n_x)
])

amp_raw_norm  = amp_raw_at_frac / amp_raw_at_frac.max()
amp_tune_norm = amp_tune / amp_tune.max()
ap_ratio      = aperture / T_tune
delta_ratio   = deltas   / T_tune

fig2, ax2 = plt.subplots(figsize=(9, 5))
fig2.suptitle("The tuning curve  —  raw amplitude vs phase indicator", fontsize=12,
              fontweight="bold")

ax2.plot(delta_ratio, amp_tune_norm,       "k--", lw=1.5, alpha=0.7,
         label="Noise-free tuning curve (Ricker pair)")
ax2.plot(ap_ratio,    amp_raw_norm,        "k-",  lw=2.0,
         label="Peak amplitude at fracture (bscan)")
ax2.plot(ap_ratio,    np.abs(sin_at_frac), "m-",  lw=2.0,
         label="|sin(theta_dom)|  at fracture")

# Mark lambda/4 (theoretical) and actual tuning peak
ax2.axvline(1.0,           color="steelblue", ls="--", lw=1.5,
            label=f"lambda/4  (T_tune = {T_tune*1e3:.0f} ps, theoretical)")
ax2.axvline(ap_tune_ratio, color="crimson",   ls="-",  lw=1.5,
            label=f"Actual tuning peak = {ap_tune_ratio:.2f} * T_tune"
                  f"  ({ap_tune_peak*1e3:.0f} ps)")

ax2.fill_betweenx([0, 1.05], 0, ap_tune_ratio,
                  alpha=0.07, color="crimson", label="Sub-tuning zone")

for lbl, ix, col in zip(pd_labels, pd_idx, pd_colors):
    ax2.axvline(ap_ratio[ix], color=col, lw=1.0, ls=":", alpha=0.9)
    ax2.text(ap_ratio[ix] + 0.05, 0.88, lbl.split("(")[0].strip(),
             color=col, fontsize=8, rotation=90, va="top")

ax2.set_xlabel("Aperture / T_tune  (= aperture / [lambda/4])", fontsize=10)
ax2.set_ylabel("Normalised value", fontsize=10)
ax2.set_xlim(0, ap_ratio.max())
ax2.set_ylim(0, 1.08)
ax2.legend(fontsize=8, loc="upper right")
ax2.set_title(
    "Amplitude peaks at the actual tuning thickness (red line), "
    "not necessarily at lambda/4 (blue dashed)\n"
    "Left of red: amplitude drops, |sin(theta_dom)| stays near 1  "
    "-> phase detects beds amplitude cannot resolve",
    fontsize=9,
)
plt.tight_layout()
plt.show()

print(f"Theoretical lambda/4      : T_tune = {T_tune*1e3:.1f} ps = T_lam / 4")
print(f"Actual tuning peak (Ricker): {ap_tune_peak*1e3:.1f} ps = {ap_tune_ratio:.2f} * T_tune "
      f"= {ap_tune_peak/T_lam:.3f} * T_lam")

# ── Figure 3: Trace overlays at the five key positions ────────────────────────
fig3, axs3 = plt.subplots(1, 5, figsize=(18, 6), sharey=True)
fig3.suptitle("Composite waveform at key apertures  (normalised per trace)", fontsize=11,
              fontweight="bold")

for ax, lbl, ix, col in zip(axs3, pd_labels, pd_idx, pd_colors):
    tr   = bscan[ix]
    norm = max(np.abs(tr).max(), 1e-9)
    ax.plot(tr / norm, t, color=col, lw=1.4)
    ax.axhline(T0[ix],               color="gray", lw=0.7, ls="--")
    ax.axhline(T0[ix] + aperture[ix], color="gray", lw=0.7, ls="--")
    ax.axvline(0, color="black", lw=0.4)
    ax.invert_yaxis()
    ax.set_ylim(T0[0] + 1.6, T0[0] - 0.6)
    ax.set_xlim(-1.3, 1.3)
    ax.set_title(lbl, fontsize=9, color=col)
    ax.set_xlabel("Norm. amp.", fontsize=8)
    if ax is axs3[0]:
        ax.set_ylabel("TWT [ns]", fontsize=9)
    ax.text(0.05, 0.02, f"{aperture[ix]*1e3:.0f} ps\n= {aperture[ix]/T_tune:.2f}*lambda/4",
            transform=ax.transAxes, fontsize=8, va="bottom")

plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# CLSSA Phase Decomposition — Constrained Least-Squares Spectral Analysis
#
# Why CLSSA instead of CWT?
#   CWT: window width scales with 1/frequency -> low-frequency components have
#        a large temporal footprint (COI), smearing thin-bed events in time.
#   CLSSA: fixed-length sliding window applied to the KERNEL (tapered sinusoids),
#          not to the data.  The inversion recovers the raw-trace spectrum with
#          uniform time resolution at all frequencies.  No COI, no edge smearing.
#
# Forward model for window centred at t_c:
#   d = G m        G[j,k] = h[j] * exp(i 2pi f_k (t_j - t_c))
#   Solution: m = (G_n^H G_n + alpha I)^{-1} G_n^H d / col_norms
#
# -90 deg phase anomaly = diagnostic signature of a sub-resolution
# low-impedance thin layer (Castagna et al. 2016).
# =============================================================================

import sys, os, importlib
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
import phase_decomposition as pd_mod
importlib.reload(pd_mod)   # pick up any edits without kernel restart

import numpy as np
import matplotlib.pyplot as plt
import time as _time

# ── CLSSA parameters ───────────────────────────────────────────────────────────
CLSSA_PARAMS = dict(
    f_min_GHz = 0.1,
    f_max_GHz = 4.0,
    n_freqs   = 60,
    win_ns    = 2.0,    # sliding window length [ns]  — ~7.5 dominant periods
    alpha     = 1e-2,   # Tikhonov regularisation
)

print("CLSSA parameters:")
for k, v in CLSSA_PARAMS.items():
    print(f"  {k:<12} = {v}")
print()

# ── Run CLSSA on each selected trace ──────────────────────────────────────────
for lbl, ix, col in zip(pd_labels, pd_idx, pd_colors):
    trace    = bscan[ix]
    ap_ratio = aperture[ix] / T_tune

    t0_run = _time.time()
    A, theta_2d_deg, pg, freqs_GHz, time_ns, theta_deg = (
        pd_mod.clssa_phase_decomposition(trace, dt, **CLSSA_PARAMS)
    )
    elapsed = _time.time() - t0_run

    fig = pd_mod.plot_clssa_decomposition(
        A, theta_2d_deg, pg, freqs_GHz, time_ns, theta_deg,
        title=(
            f"CLSSA — {lbl}  |  "
            f"aperture = {aperture[ix]*1e3:.0f} ps = {ap_ratio:.2f}*T_tune  "
            f"({elapsed:.1f} s)"
        ),
        highlight_neg90=True,
    )
    plt.show()

# ── Side-by-side comparison: CWT vs CLSSA on the lambda/4 trace ───────────────
print("\nComparing CWT vs CLSSA on the Rayleigh-limit trace (lambda/4)...")
ix_comp = pd_idx[3]   # lambda/4 trace

# CWT
A_cwt, th_cwt, pg_cwt, fq_cwt, tn_cwt, td_cwt = pd_mod.phase_decomposition(
    bscan[ix_comp], dt,
    f_min_GHz=0.5, f_max_GHz=4.0, n_scales=60, reflect_pad=True,
)
# CLSSA
A_cl, th_cl, pg_cl, fq_cl, tn_cl, td_cl = pd_mod.clssa_phase_decomposition(
    bscan[ix_comp], dt, **CLSSA_PARAMS
)

fig2, axes2 = plt.subplots(2, 3, figsize=(15, 10), sharey=True)
fig2.suptitle(
    f"CWT vs CLSSA  —  {pd_labels[3]}  "
    f"(aperture = {aperture[ix_comp]*1e3:.0f} ps = {aperture[ix_comp]/T_tune:.2f}*T_tune)",
    fontsize=12, fontweight="bold",
)

t0v, t1v = tn_cwt[0], tn_cwt[-1]

def _fill_row(axes_row, A, th, pg, fq, tn, td, method):
    db_clip = 40.0
    f0, f1   = fq[0], fq[-1]
    ext_sp   = [f0, f1, t1v, t0v]

    A_dB = np.clip(20*np.log10(A / (A.max()+1e-30) + 1e-30), -db_clip, 0)
    im0 = axes_row[0].imshow(A_dB.T, aspect="auto", origin="upper",
                              extent=ext_sp, cmap="jet",
                              vmin=-db_clip, vmax=0, interpolation="bilinear")
    fig2.colorbar(im0, ax=axes_row[0], fraction=0.046, pad=0.04).set_label("dB", fontsize=8)
    axes_row[0].set_title(f"{method} — Amplitude", fontsize=10)
    axes_row[0].set_xlabel("Frequency [GHz]", fontsize=9)
    axes_row[0].set_ylabel("TWT [ns]", fontsize=9)

    im1 = axes_row[1].imshow(th.T, aspect="auto", origin="upper",
                              extent=ext_sp, cmap="RdBu",
                              vmin=-180, vmax=180, interpolation="nearest")
    fig2.colorbar(im1, ax=axes_row[1], fraction=0.046, pad=0.04).set_label("deg", fontsize=8)
    axes_row[1].set_title(f"{method} — Phase", fontsize=10)
    axes_row[1].set_xlabel("Frequency [GHz]", fontsize=9)

    pg_c = np.percentile(np.abs(pg), 99)
    im2 = axes_row[2].imshow(pg.T, aspect="auto", origin="upper",
                              extent=[td[0], td[-1], t1v, t0v],
                              cmap="RdBu", vmin=-pg_c, vmax=pg_c,
                              interpolation="bilinear")
    fig2.colorbar(im2, ax=axes_row[2], fraction=0.046, pad=0.04).set_label("Amp", fontsize=8)
    axes_row[2].set_title(f"{method} — Phase Gather", fontsize=10)
    axes_row[2].set_xlabel("Phase [deg]", fontsize=9)
    axes_row[2].axvline(-90, color="magenta", lw=1.5, ls="--")
    axes_row[2].axvline( 90, color="lime",    lw=1.0, ls="--")
    axes_row[2].axvspan(-105, -75, color="magenta", alpha=0.15)
    axes_row[2].set_xticks([-180, -90, 0, 90, 180])

_fill_row(axes2[0], A_cwt, th_cwt, pg_cwt, fq_cwt, tn_cwt, td_cwt, "CWT")
_fill_row(axes2[1], A_cl,  th_cl,  pg_cl,  fq_cl,  tn_cl,  td_cl,  "CLSSA")

plt.tight_layout()
plt.show()

# ── Dominant-phase diagnostic for each selected trace ─────────────────────────
# The phase gather's inclined bands reflect carrier-frequency rotation (dtheta/dt ~ 2pi*f_c).
# plot_dominant_phase collapses the 2-D phase gather to a 1-D line:
#   theta_dom(t) = arctan2(I(t), R(t))
# A phase jump TO -90 deg at the event time = thin-bed / sub-resolution signature.

for lbl, ix, col in zip(pd_labels, pd_idx, pd_colors):
    trace = bscan[ix]
    A, theta_2d_deg, pg, freqs_GHz, time_ns, theta_deg = (
        pd_mod.clssa_phase_decomposition(trace, dt, **CLSSA_PARAMS)
    )
    fig = pd_mod.plot_dominant_phase(
        A, theta_2d_deg, pg, freqs_GHz, time_ns, theta_deg,
        title=f"Dominant phase — {lbl}  |  ap={aperture[ix]*1e3:.0f} ps = {aperture[ix]/T_tune:.2f}*T_tune",
    )
    plt.show()